# Module 3 — Genie space over CVE / KEV / STIG / Assets

**Time**: ~25 minutes

**For analysts**: Genie lets you ask English questions over Databricks tables and get back SQL answers + charts. We're going to wire one up over the CVE / KEV / asset tables you saw in Module 1.

**For engineers**: Genie is configured by **(a)** the table set, **(b)** instructions / glossary you teach it, and **(c)** registered SQL functions for parametric lookups. We'll use the SDK to declare all three programmatically — no clicking through the UI.

We'll also export a Genie-generated SQL query at the end and use it in **Module 6 — vibe-code demo**.

## 1. Register helper SQL functions

These show up as named tools in Genie. The model uses them when a question maps cleanly to a parametrized lookup.

In [ ]:
%sql
CREATE OR REPLACE FUNCTION disa_workshop.threat_intel.lookup_cve(cve STRING)
RETURNS TABLE
COMMENT 'Lookup a single CVE by ID and return its CVSS score, description, KEV status, and date added to KEV.'
RETURN
  SELECT
    c.cve_id,
    c.description,
    c.cvss_score,
    c.cvss_severity,
    k.cveID IS NOT NULL AS in_kev,
    k.dateAdded AS kev_date_added,
    k.requiredAction AS kev_required_action
  FROM disa_workshop.threat_intel.cves c
  LEFT JOIN disa_workshop.threat_intel.kev_catalog k ON c.cve_id = k.cveID
  WHERE c.cve_id = lookup_cve.cve;

In [ ]:
%sql
CREATE OR REPLACE FUNCTION disa_workshop.threat_intel.assets_for_product(vendor_name STRING, product_name STRING)
RETURNS TABLE
COMMENT 'Return all assets running a given vendor and product. Use to scope CVE impact to known DoD assets.'
RETURN
  SELECT asset_id, hostname, environment, owner_org, last_patched
  FROM disa_workshop.threat_intel.affected_assets
  WHERE vendor = vendor_name AND product = product_name;

## 2. Create the Genie space programmatically

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

GENIE_INSTRUCTIONS = """
You are a cyber threat intelligence analyst assistant for DISA.

Vocabulary:
- KEV = CISA Known Exploited Vulnerabilities catalog. Entries here are actively exploited in the wild.
- CVE = Common Vulnerabilities and Exposures, format CVE-YYYY-NNNNN.
- CVSS = severity score 0.0 - 10.0. Critical >= 9.0, High >= 7.0.
- Affected assets are joined on vendor + product columns.
- Environments: NIPRNet (unclass), SIPRNet (secret), JWICS (TS).

When asked about "actively exploited" or "high-priority" CVEs, prefer the KEV catalog.
When asked about asset exposure, join CVEs/KEV to affected_assets via vendor + product.
Always cite CVE IDs and date ranges in your answer.
""".strip()

SAMPLE_QUESTIONS = [
    "Which KEV-listed CVEs were added in the last 30 days?",
    "Show me CVEs with CVSS score above 9 affecting Microsoft products in our asset inventory.",
    "Top 10 vendors by KEV catalog entry count.",
    "Which of our SIPRNet hosts are running products affected by CVEs added to KEV this year?",
    "How many CVEs have we seen in 2025 with attack vector 'NETWORK' and severity HIGH or CRITICAL?",
    "Which DoD organizations own the most assets exposed to KEV vulnerabilities?",
    "Compare the count of CVEs published in 2024 vs 2025.",
    "Show me Cisco IOS XE CVEs and the assets running Cisco IOS XE.",
    "Which assets have not been patched in over 60 days?",
    "What is the median CVSS score of KEV-listed CVEs?",
]

for q in SAMPLE_QUESTIONS:
    print(f"  {q}")

In [ ]:
# Create the Genie space.
# In a notebook, this prints the URL to open in a new tab.
# (The Genie SDK API surface is evolving — check
# `databricks-ai-dev-kit:databricks-genie` for the current canonical pattern.)

space_config = {
    "display_name": "DISA Threat Intel",
    "description": "Cyber threat intelligence over CVE, KEV, MITRE ATT&CK, and DoD asset inventory.",
    "warehouse_id": w.config.warehouse_id or w.warehouses.list()[0].id,
    "datasets": [
        {"table": "disa_workshop.threat_intel.kev_catalog"},
        {"table": "disa_workshop.threat_intel.cves"},
        {"table": "disa_workshop.threat_intel.affected_assets"},
        {"table": "disa_workshop.threat_intel.attack_techniques"},
        {"table": "disa_workshop.threat_intel.attack_groups"},
    ],
    "instructions": GENIE_INSTRUCTIONS,
    "sample_questions": SAMPLE_QUESTIONS,
    "functions": [
        "disa_workshop.threat_intel.lookup_cve",
        "disa_workshop.threat_intel.assets_for_product",
    ],
}

print("Genie space configuration:")
import json
print(json.dumps(space_config, indent=2))
print("\n>>> In the workspace, click 'Genie' in the sidebar > New space, paste this config, and save.")
print(">>> Or use the Genie REST API: PUT /api/2.0/genie/spaces/<space_id>")

## 3. Walk through the 10 sample questions live

Open the Genie space you just created and ask the 10 sample questions one at a time. Discuss as a group:
- Did Genie pick the right tables?
- Did it use the SQL functions or write inline SQL?
- Which questions need clearer instructions or vocabulary?

**For engineers**: click "View SQL" on any answer to see what Genie generated. We'll use one of these in Module 6.

## 4. Save Genie SQL for Module 6

Pick one of the sample questions, copy the SQL from the Genie UI, and paste it into the cell below. We'll use it later to vibe-code a React chart for the app.

In [ ]:
# Paste a Genie-generated SQL statement here, run it, confirm the result, then copy it
# (along with the column schema) into prompts/vibe_code_component.md for Module 6.

GENIE_SQL = """
SELECT vendor, COUNT(*) AS cve_count
FROM disa_workshop.threat_intel.kev_catalog
GROUP BY vendor
ORDER BY cve_count DESC
LIMIT 10
"""

result = spark.sql(GENIE_SQL)
result.printSchema()
result.display()

**Next**: **Module 4 — `notebooks/04_knowledge_assistant.ipynb`**.